In [1]:
html = """<!DOCTYPE html>
<html>
<head>
    <title>Course Catalog</title>
</head>
<body>
<div class="container">
    <h1>Online Courses</h1>

    <div class="course" data-id="C101">
        <h2 class="course-title">Python for Beginners</h2>
        <p class="duration">8 weeks</p>
        <p class="level">Beginner</p>
    </div>

    <div class="course" data-id="C102">
        <h2 class="course-title">Advanced Web Development</h2>
        <p class="duration">10 weeks</p>
        <p class="level">Advanced</p>
    </div>

    <div class="course" data-id="C103">
        <h2 class="course-title">Data Analysis with Pandas</h2>
        <p class="duration">6 weeks</p>
        <p class="level">Intermediate</p>
    </div>
</div>
</body>
</html>"""

In [2]:
from bs4 import BeautifulSoup

soup=BeautifulSoup(html,"html.parser")

titles=soup.find_all('h2',class_='course-title')

for i , title in enumerate(titles):
    print(f"courde title {i+1}: ",title.text)

courde title 1:  Python for Beginners
courde title 2:  Advanced Web Development
courde title 3:  Data Analysis with Pandas


In [3]:
from bs4 import BeautifulSoup

soup=BeautifulSoup(html,"html.parser")

paras=soup.find_all('p',class_='duration')

for i, para in enumerate(paras):
    print(f"Course Duration {i+1}: ",para.text)

Course Duration 1:  8 weeks
Course Duration 2:  10 weeks
Course Duration 3:  6 weeks


In [4]:
from bs4 import BeautifulSoup

soup= BeautifulSoup(html, 'html.parser')

levels=soup.find_all('p','level')

for i , level in enumerate(levels):
    print(f"Course level {i+1}: ",level.text)

Course level 1:  Beginner
Course level 2:  Advanced
Course level 3:  Intermediate


In [ ]:
from bs4 import BeautifulSoup

soup= BeautifulSoup(html, 'html.parser')

levels=soup.find('p','level')

for i , level in enumerate(levels):
    print(f"Course level {i+1}: ",level.text)


# find select only one which match the condition and 
# and find_all give all matches as output

Course level 1:  Beginner


In [11]:
pip install scrapy

In [13]:

from scrapy import Selector

selector = Selector(text = html)

titles=selector.css("h2.course-title::text").getall()

for i , title in enumerate(titles,start=1):
    print(f"Course Title {i}: {title.strip()}")

AttributeError: module 'scrapy' has no attribute 'Spider' (consider renaming 'd:\WebScraping\scrapy.py' if it has the same name as a library you intended to import)

In [27]:
# Atrribute and text extraction

html2  = '''<html>
<body>
<div class="article" data-article-id="101">
    <h2 class="title">Intro to Machine Learning</h2>
    <span class="author">Alice</span>
</div>
<div class="article" data-article-id="102">
    <h2 class="title">Web Scraping Guide</h2>
    <span class="author">Bob</span>
</div>
</body>
</html>'''

In [30]:
# Attribute and text extraction

soup = BeautifulSoup(html2, "lxml")

articles = []

for article in soup.select("div.article"):
    articles.append({
        "title": article.select_one("h2.title").get_text(strip=True),
        "author": article.select_one("span.author").get_text(strip=True),
        "data-article-id": article.get("data-article-id")
    })

print(articles)

[{'title': 'Intro to Machine Learning', 'author': 'Alice', 'data-article-id': '101'}, {'title': 'Web Scraping Guide', 'author': 'Bob', 'data-article-id': '102'}]


In [31]:
html3 = '''<html>
<body>
<div id="page-loaded"></div>
<div class="comment">Great tutorial!</div>
<div class="comment">Very helpful, thanks!</div>
</body>
</html>
'''

In [33]:
# wait and dynamic content
from urllib.parse import quote
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

driver= webdriver.Chrome()

driver.get("data:text/html;charset=utf-8," + quote(html3))

WebDriverWait(driver, timeout=10).until(
    EC.presence_of_element_located((By.ID, "page-loaded"))
)

comments = [
    element.text
    for element in driver.find_elements(By.CLASS_NAME, "comment")
]

print(comments)

driver.quit()


['Great tutorial!', 'Very helpful, thanks!']


In [36]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException

def find_filter_container(driver, id=None, class_name=None):
    if id is not None:
        return driver.find_element(By.ID, id)

    if class_name is not None:
        return driver.find_element(By.CLASS_NAME, class_name)

    raise ValueError("Provide either id or class_name")


def main():
    driver=webdriver.Chrome()

    try:
        driver.get("https://ranveerbrar.com/recipes/")

        result=find_filter_container(
            driver,
            id="recipes_filter_results"
        )
        print(result.text)

    except NoSuchElementException:
        print("the requested element is not found")
    
    finally:
        driver.quit()

if __name__=="__main__":
    main()

the requested element is not found


In [38]:
from selenium import webdriver
from bs4 import BeautifulSoup

url = "https://ranveerbrar.com/recipes/"

driver = webdriver.Chrome()

try:
    driver.get(url)

    soup = BeautifulSoup(driver.page_source, "lxml")

    print("Page URL:", driver.current_url)
    print("Page title:", driver.title)
    print("HTML length:", len(driver.page_source))

    print("\n--- Element counts ---")
    for selector in [
        "h1", "h2", "h3", "h4",
        "a[href]", "img",
        "#recipes_filter_results",
        ".course", "article"
    ]:
        print(selector, ":", len(soup.select(selector)))

    print("\n--- First 20 links ---")
    for link in soup.select("a[href]")[:20]:
        print(
            "TEXT:", link.get_text(" ", strip=True)[:100],
            "| URL:", link.get("href")
        )

    print("\n--- First 15 headings ---")
    for heading in soup.select("h1, h2, h3, h4")[:15]:
        print(heading.name, ":", heading.get_text(" ", strip=True))

    with open("debug_page.html", "w", encoding="utf-8") as file:
        file.write(driver.page_source)

    print("\nSaved rendered HTML to debug_page.html")

finally:
    driver.quit()

Page URL: https://ranveerbrar.com/recipes/
Page title: 1000+ Cooking Recipes From Ranveer Brar | Healthy Veg, Non Veg and Vegan Recipes
HTML length: 1078517

--- Element counts ---
h1 : 2
h2 : 1
h3 : 3
h4 : 0
a[href] : 1669
img : 25
#recipes_filter_results : 0
.course : 0
article : 0

--- First 20 links ---
TEXT: Skip to content | URL: #content
TEXT:  | URL: https://ranveerbrar.com/
TEXT: Ranveer Brar | URL: https://ranveerbrar.com/
TEXT: Home | URL: https://ranveerbrar.com/
TEXT: About | URL: https://ranveerbrar.com/about/
TEXT: Recipes | URL: https://ranveerbrar.com/recipes/
TEXT: Shop | URL: https://ranveerbrar.com/product/a-journey-through-india/
TEXT: Blog | URL: https://ranveerbrar.com/blog/
TEXT: Press | URL: https://ranveerbrar.com/press/
TEXT: Contact Us | URL: https://ranveerbrar.com/contact/
TEXT: RECIPES | URL: https://ranveerbrar.com/recipes/
TEXT:  | URL: #
TEXT:  | URL: https://ranveerbrar.com/cart/
TEXT: Email us | URL: mailto:contact@ranveerbrar.com
TEXT: Submit a Reci

In [ ]:
# question 1 extract recipes titles
from selenium import webdriver
from bs4 import BeautifulSoup
from selenium.webdriver.support.ui import WebDriverWait


def main():
    driver = webdriver.Chrome()

    try:
        driver.get("https://ranveerbrar.com/recipes/")

        # Wait for the document to finish loading.
        WebDriverWait(driver, 15).until(
            lambda d: d.execute_script(
                "return document.readyState"
            ) == "complete"
        )

        # Get the rendered HTML.
        html = driver.page_source

        # Parse the HTML.
        soup = BeautifulSoup(html, "lxml")

        print("Page title:", driver.title)

        # Inspect headings to discover the actual title selectors.
        headings = soup.select("h1, h2, h3, h4, h5, h6")

        print("Headings found:", len(headings))
        print("\nRecipe titles / headings:")

        for heading in headings:
            title = heading.get_text(" ", strip=True)

            if title:
                print(title)

    finally:
        driver.quit()


if __name__ == "__main__":
    main()


Page title: 1000+ Cooking Recipes From Ranveer Brar | Healthy Veg, Non Veg and Vegan Recipes
Headings found: 6

Recipe titles / headings:
Ranveer's Recipes
Filter By
About Me
My books on Amazon.com
Follow & Connect
Chef Ranveer AI


In [41]:

from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse


BASE_URL = "https://ranveerbrar.com/"
RECIPES_URL = urljoin(BASE_URL, "recipes/")


def extract_recipe_links(soup, page_url):
    recipe_links = []
    seen_urls = set()

    for link in soup.select("a[href]"):
        href = link.get("href")

        if not href:
            continue

        # Convert relative URLs into absolute URLs.
        full_url = urljoin(page_url, href)

        # Keep links belonging to Ranveer Brar's website.
        domain = urlparse(full_url).netloc.lower()

        if domain not in {
            "ranveerbrar.com",
            "www.ranveerbrar.com"
        }:
            continue

        # Avoid duplicate URLs.
        if full_url in seen_urls:
            continue

        seen_urls.add(full_url)

        # Store the URL.
        recipe_links.append(full_url)

    return recipe_links


def main():
    driver = webdriver.Chrome()

    try:
        driver.get(RECIPES_URL)

        WebDriverWait(driver, 15).until(
            lambda d: d.execute_script(
                "return document.readyState"
            ) == "complete"
        )

        soup = BeautifulSoup(
            driver.page_source,
            "lxml"
        )

        recipe_links = extract_recipe_links(
            soup,
            driver.current_url
        )

        print("Total unique internal links:", len(recipe_links))

        for i, url in enumerate(recipe_links, start=1):
            print(f"{i}. {url}")

    finally:
        driver.quit()


if __name__ == "__main__":
    main()


Total unique internal links: 1647
1. https://ranveerbrar.com/recipes/#content
2. https://ranveerbrar.com/
3. https://ranveerbrar.com/about/
4. https://ranveerbrar.com/recipes/
5. https://ranveerbrar.com/product/a-journey-through-india/
6. https://ranveerbrar.com/blog/
7. https://ranveerbrar.com/press/
8. https://ranveerbrar.com/contact/
9. https://ranveerbrar.com/cart/
10. https://ranveerbrar.com/submit-recipe/
11. https://ranveerbrar.com/recipes
12. https://ranveerbrar.com/user-recipes/
13. https://ranveerbrar.com/recipe-tag/recipe-of-the-week/
14. https://ranveerbrar.com/special-recipes/monsoon-recipes/
15. https://ranveerbrar.com/special-recipes/party-snacks/
16. https://ranveerbrar.com/special-recipes/summer-recipes/
17. https://ranveerbrar.com/special-recipes/festival-special/
18. https://ranveerbrar.com/recipe_veg/breakfast/
19. https://ranveerbrar.com/recipe_veg/main-course/
20. https://ranveerbrar.com/recipe_nonveg/noodles-and-pastas/
21. https://ranveerbrar.com/recipe_nonveg/s

In [44]:
from __future__ import annotations

import csv
import json
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse, urldefrag

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait


BASE_URL = "https://ranveerbrar.com/"
START_URL = urljoin(BASE_URL, "recipes/")
OUTPUT_DIR = Path("scraped_data")
LISTINGS_CSV = OUTPUT_DIR / "recipe_listings.csv"
DETAILS_JSON = OUTPUT_DIR / "recipe_details.json"

PAGE_LOAD_TIMEOUT = 30
WAIT_TIMEOUT = 15
REQUEST_DELAY = 0.5
MAX_PAGES = 500

ALLOWED_HOSTS = {"ranveerbrar.com", "www.ranveerbrar.com"}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def normalize_url(url: str, base_url: str = BASE_URL) -> str:
    absolute = urljoin(base_url, url.strip())
    absolute, _ = urldefrag(absolute)
    parsed = urlparse(absolute)

    if parsed.scheme not in {"http", "https"}:
        return ""

    hostname = (parsed.hostname or "").lower()
    if hostname not in ALLOWED_HOSTS:
        return ""

    return parsed._replace(
        scheme="https",
        netloc="ranveerbrar.com",
        fragment="",
    ).geturl().rstrip("/")


def clean_text(value: str | None) -> str:
    return " ".join((value or "").split())


def get_soup(driver: webdriver.Chrome, url: str) -> BeautifulSoup:
    driver.get(url)

    try:
        WebDriverWait(driver, WAIT_TIMEOUT).until(
            lambda d: d.execute_script(
                "return document.readyState"
            ) in {"interactive", "complete"}
        )
    except TimeoutException:
        pass

    return BeautifulSoup(driver.page_source, "lxml")


def first_text(soup: BeautifulSoup, selectors: list[str]) -> str:
    for selector in selectors:
        element = soup.select_one(selector)
        if element:
            text = clean_text(element.get_text(" ", strip=True))
            if text:
                return text
    return ""


def get_meta_content(soup: BeautifulSoup, *selectors: str) -> str:
    for selector in selectors:
        element = soup.select_one(selector)
        if element:
            content = clean_text(element.get("content"))
            if content:
                return content
    return ""


def extract_page_metadata(
    soup: BeautifulSoup,
    page_url: str,
) -> dict:
    title = ""

    if soup.title:
        title = clean_text(soup.title.get_text(" ", strip=True))

    if not title:
        title = first_text(soup, ["h1"])

    description = get_meta_content(
        soup,
        'meta[name="description"]',
        'meta[property="og:description"]',
        'meta[name="twitter:description"]',
    )

    canonical_element = soup.select_one('link[rel~="canonical"][href]')
    canonical_url = (
        normalize_url(canonical_element.get("href"), page_url)
        if canonical_element
        else ""
    )

    return {
        "title": title,
        "description": description,
        "canonical_url": canonical_url,
    }


def extract_images(
    soup: BeautifulSoup,
    page_url: str,
) -> list[dict]:
    images = []
    seen = set()

    for image in soup.select("img"):
        candidates = [
            image.get("src"),
            image.get("data-src"),
            image.get("data-lazy-src"),
            image.get("data-original"),
            image.get("data-lazy"),
            image.get("data-url"),
        ]

        srcset = image.get("srcset") or image.get("data-srcset")
        if srcset:
            first_candidate = srcset.split(",")[0].strip().split()
            if first_candidate:
                candidates.insert(0, first_candidate[0])

        image_url = ""

        for candidate in candidates:
            if candidate and not candidate.startswith("data:"):
                image_url = urljoin(page_url, candidate.strip())
                break

        if not image_url or image_url in seen:
            continue

        seen.add(image_url)

        images.append({
            "url": image_url,
            "alt": clean_text(image.get("alt")),
            "loading": clean_text(image.get("loading")),
            "lazy_loading": any(
                image.get(attribute)
                for attribute in (
                    "data-src",
                    "data-lazy-src",
                    "data-original",
                    "data-lazy",
                    "data-srcset",
                )
            ) or image.get("loading") == "lazy",
            "src": clean_text(image.get("src")),
            "srcset": clean_text(
                image.get("srcset") or image.get("data-srcset")
            ),
        })

    return images


def is_recipe_candidate(url: str, label: str = "") -> bool:
    path = urlparse(url).path.lower().strip("/")

    if not path:
        return False

    excluded_segments = {
        "recipes",
        "category",
        "categories",
        "author",
        "authors",
        "tag",
        "tags",
        "page",
        "about",
        "contact",
        "privacy-policy",
        "terms-and-conditions",
        "search",
        "cart",
        "checkout",
        "my-account",
    }

    segments = [segment for segment in path.split("/") if segment]

    if any(segment in excluded_segments for segment in segments):
        return False

    if len(segments) >= 2 and segments[0] in {
        "recipe",
        "recipes",
    }:
        return True

    return False


def extract_recipe_listing(
    soup: BeautifulSoup,
    page_url: str,
) -> list[dict]:
    recipes = []
    seen = set()

    selectors = [
        "article a[href]",
        ".recipe a[href]",
        ".recipes a[href]",
        ".recipe-card a[href]",
        ".post a[href]",
        ".entry-title a[href]",
        "h2 a[href]",
        "h3 a[href]",
        "h4 a[href]",
    ]

    candidates = []
    for selector in selectors:
        candidates.extend(soup.select(selector))

    if not candidates:
        candidates = soup.select("a[href]")

    for anchor in candidates:
        href = anchor.get("href")
        if not href:
            continue

        recipe_url = normalize_url(href, page_url)
        if not recipe_url or recipe_url in seen:
            continue

        if not is_recipe_candidate(recipe_url):
            continue

        seen.add(recipe_url)

        container = anchor.find_parent(
            ["article", "li", "div", "section"]
        ) or anchor.parent

        title = clean_text(anchor.get_text(" ", strip=True))

        if not title and container:
            title_element = container.select_one(
                "h1, h2, h3, h4, .entry-title, .post-title"
            )
            if title_element:
                title = clean_text(
                    title_element.get_text(" ", strip=True)
                )

        image_element = container.select_one("img") if container else None
        image_url = ""

        if image_element:
            for attribute in (
                "src",
                "data-src",
                "data-lazy-src",
                "data-original",
            ):
                value = image_element.get(attribute)
                if value and not value.startswith("data:"):
                    image_url = urljoin(page_url, value)
                    break

        recipes.append({
            "title": title,
            "url": recipe_url,
            "image_url": image_url,
            "image_alt": (
                clean_text(image_element.get("alt"))
                if image_element
                else ""
            ),
        })

    return recipes


def extract_category_links(
    soup: BeautifulSoup,
    page_url: str,
) -> list[dict]:
    categories = []
    seen = set()

    selectors = [
        ".recipe-category a[href]",
        ".recipe-categories a[href]",
        ".category a[href]",
        ".categories a[href]",
        ".widget_categories a[href]",
        "a[rel~='category']",
        "a[href*='/category/']",
        "a[href*='/recipe-category/']",
    ]

    for selector in selectors:
        for anchor in soup.select(selector):
            href = anchor.get("href")
            label = clean_text(anchor.get_text(" ", strip=True))

            if not href or not label:
                continue

            category_url = normalize_url(href, page_url)
            if not category_url or category_url in seen:
                continue

            seen.add(category_url)
            categories.append({
                "label": label,
                "url": category_url,
            })

    return categories


def find_next_page(
    soup: BeautifulSoup,
    page_url: str,
) -> str | None:
    selectors = [
        "a[rel='next'][href]",
        ".pagination a.next[href]",
        ".nav-links a.next[href]",
        "a.next.page-numbers[href]",
        "a.next[href]",
        "a[aria-label*='Next'][href]",
        "a[aria-label*='next'][href]",
    ]

    for selector in selectors:
        element = soup.select_one(selector)
        if element:
            url = normalize_url(element.get("href"), page_url)
            if url and url != normalize_url(page_url):
                return url

    for anchor in soup.select("a[href]"):
        label = clean_text(anchor.get_text(" ", strip=True)).lower()
        rel = anchor.get("rel") or []

        if isinstance(rel, str):
            rel = rel.split()

        if (
            "next" in [item.lower() for item in rel]
            or label in {"next", "next page", "older posts", "→", "›", "»"}
        ):
            url = normalize_url(anchor.get("href"), page_url)
            if url and url != normalize_url(page_url):
                return url

    return None


def extract_section_text(
    soup: BeautifulSoup,
    headings: list[str],
) -> str:
    heading_words = [word.lower() for word in headings]

    for heading in soup.find_all(
        ["h1", "h2", "h3", "h4", "h5", "h6"]
    ):
        heading_text = clean_text(
            heading.get_text(" ", strip=True)
        ).lower()

        if not any(word in heading_text for word in heading_words):
            continue

        collected = []
        for sibling in heading.find_all_next():
            if sibling.name in {"h1", "h2", "h3", "h4", "h5", "h6"}:
                break

            if sibling.name in {"script", "style", "nav", "footer"}:
                continue

            if sibling.name in {"li", "p"}:
                text = clean_text(
                    sibling.get_text(" ", strip=True)
                )
                if text:
                    collected.append(text)

        if collected:
            return "\n".join(dict.fromkeys(collected))

    return ""


def extract_ingredients(soup: BeautifulSoup) -> list[str]:
    ingredients = []

    selectors = [
        ".wprm-recipe-ingredients li",
        ".tasty-recipes-ingredients li",
        ".recipe-ingredients li",
        ".ingredients li",
        "[itemprop='recipeIngredient']",
        ".ingredients-container li",
    ]

    for selector in selectors:
        elements = soup.select(selector)
        if elements:
            ingredients = [
                clean_text(element.get_text(" ", strip=True))
                for element in elements
            ]
            ingredients = [item for item in ingredients if item]
            if ingredients:
                return list(dict.fromkeys(ingredients))

    section = extract_section_text(
        soup,
        ["ingredients", "what you need"],
    )

    if section:
        ingredients = [
            clean_text(line.lstrip("•-* "))
            for line in section.splitlines()
        ]

    return [item for item in ingredients if item]


def extract_instructions(soup: BeautifulSoup) -> list[str]:
    selectors = [
        ".wprm-recipe-instructions li",
        ".tasty-recipes-instructions li",
        ".recipe-instructions li",
        ".instructions li",
        ".directions li",
        "[itemprop='recipeInstructions'] li",
    ]

    for selector in selectors:
        elements = soup.select(selector)
        if elements:
            instructions = [
                clean_text(element.get_text(" ", strip=True))
                for element in elements
            ]
            instructions = [item for item in instructions if item]
            if instructions:
                return list(dict.fromkeys(instructions))

    section = extract_section_text(
        soup,
        ["instructions", "method", "directions", "preparation"],
    )

    return [
        clean_text(line.lstrip("•-* "))
        for line in section.splitlines()
        if clean_text(line.lstrip("•-* "))
    ]


def extract_recipe_times(soup: BeautifulSoup) -> dict:
    def value_for(selectors: list[str]) -> str:
        for selector in selectors:
            element = soup.select_one(selector)
            if element:
                value = clean_text(
                    element.get("content")
                    or element.get_text(" ", strip=True)
                )
                if value:
                    return value
        return ""

    return {
        "prep_time": value_for([
            "[itemprop='prepTime']",
            ".wprm-recipe-prep_time",
            ".tasty-recipes-prep-time",
            ".prep-time",
        ]),
        "cook_time": value_for([
            "[itemprop='cookTime']",
            ".wprm-recipe-cook_time",
            ".tasty-recipes-cook-time",
            ".cook-time",
        ]),
        "total_time": value_for([
            "[itemprop='totalTime']",
            ".wprm-recipe-total_time",
            ".tasty-recipes-total-time",
            ".total-time",
        ]),
    }


def extract_servings(soup: BeautifulSoup) -> str:
    selectors = [
        "[itemprop='recipeYield']",
        ".wprm-recipe-servings",
        ".tasty-recipes-yield",
        ".recipe-servings",
        ".servings",
        ".yield",
    ]

    return first_text(soup, selectors) or next(
        (
            clean_text(element.get("content"))
            for selector in selectors
            for element in soup.select(selector)
            if element.get("content")
        ),
        "",
    )


def extract_full_recipe(
    driver: webdriver.Chrome,
    recipe_url: str,
) -> dict:
    result = {
        "url": recipe_url,
        "title": "",
        "description": "",
        "canonical_url": "",
        "images": [],
        "ingredients": [],
        "instructions": [],
        "times": {
            "prep_time": "",
            "cook_time": "",
            "total_time": "",
        },
        "servings": "",
        "categories": [],
        "status": "ok",
        "error": "",
    }

    try:
        soup = get_soup(driver, recipe_url)

        metadata = extract_page_metadata(soup, recipe_url)
        result.update(metadata)

        result["title"] = (
            first_text(
                soup,
                [
                    "h1.entry-title",
                    "h1.recipe-title",
                    "h1",
                    "[itemprop='name']",
                ],
            )
            or metadata["title"]
        )

        result["images"] = extract_images(soup, recipe_url)
        result["ingredients"] = extract_ingredients(soup)
        result["instructions"] = extract_instructions(soup)
        result["times"] = extract_recipe_times(soup)
        result["servings"] = extract_servings(soup)
        result["categories"] = extract_category_links(
            soup,
            recipe_url,
        )

    except (TimeoutException, WebDriverException, ValueError) as exc:
        result["status"] = "error"
        result["error"] = str(exc)

    return result


def save_listings_csv(recipes: list[dict]) -> None:
    fieldnames = ["title", "url", "image_url", "image_alt"]

    with LISTINGS_CSV.open(
        "w",
        newline="",
        encoding="utf-8-sig",
    ) as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(recipes)


def save_details_json(recipes: list[dict]) -> None:
    with DETAILS_JSON.open("w", encoding="utf-8") as file:
        json.dump(
            recipes,
            file,
            ensure_ascii=False,
            indent=2,
        )


def create_driver() -> webdriver.Chrome:
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1440,1200")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.page_load_strategy = "normal"

    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT)
    return driver


def scrape() -> None:
    driver = create_driver()

    listings_by_url: dict[str, dict] = {}
    details_by_url: dict[str, dict] = {}
    visited_pages: set[str] = set()
    visited_recipes: set[str] = set()

    current_page = normalize_url(START_URL)

    try:
        page_count = 0

        while current_page and current_page not in visited_pages:
            if page_count >= MAX_PAGES:
                break

            page_count += 1
            visited_pages.add(current_page)

            try:
                soup = get_soup(driver, current_page)
            except (TimeoutException, WebDriverException):
                break

            for recipe in extract_recipe_listing(soup, current_page):
                recipe_url = normalize_url(recipe["url"])
                if not recipe_url:
                    continue

                if recipe_url not in listings_by_url:
                    listings_by_url[recipe_url] = {
                        **recipe,
                        "url": recipe_url,
                    }

            next_page = find_next_page(soup, current_page)
            if not next_page or next_page in visited_pages:
                break

            current_page = next_page
            time.sleep(REQUEST_DELAY)

        save_listings_csv(list(listings_by_url.values()))

        for recipe_url in listings_by_url:
            if recipe_url in visited_recipes:
                continue

            visited_recipes.add(recipe_url)
            details = extract_full_recipe(driver, recipe_url)
            details_by_url[recipe_url] = details
            time.sleep(REQUEST_DELAY)

        save_details_json(list(details_by_url.values()))

        print(f"Pages visited: {len(visited_pages)}")
        print(f"Unique recipes: {len(listings_by_url)}")
        print(f"CSV saved to: {LISTINGS_CSV.resolve()}")
        print(f"JSON saved to: {DETAILS_JSON.resolve()}")

    finally:
        driver.quit()


if __name__ == "__main__":
    scrape()

Pages visited: 1
Unique recipes: 0
CSV saved to: D:\WebScraping\scraped_data\recipe_listings.csv
JSON saved to: D:\WebScraping\scraped_data\recipe_details.json


In [43]:
def is_recipe_candidate(url: str, from_recipe_card: bool = False) -> bool:
    path = urlparse(url).path.strip("/")
    segments = [segment.lower() for segment in path.split("/") if segment]

    if not segments:
        return False

    excluded_paths = {
        "recipes",
        "category",
        "categories",
        "recipe-category",
        "tag",
        "author",
        "about",
        "contact",
        "privacy-policy",
        "terms-and-conditions",
        "search",
        "cart",
        "checkout",
        "my-account",
    }

    if len(segments) == 1 and segments[0] in excluded_paths:
        return False

    if segments[0] in {"category", "categories", "tag", "author"}:
        return False

    if segments[0] in {"recipe", "recipes", "recipe-category"}:
        return len(segments) >= 2 and segments[0] != "recipe-category"

    return from_recipe_card and len(segments) >= 1


def extract_recipe_listing(
    soup: BeautifulSoup,
    page_url: str,
) -> list[dict]:
    recipes = []
    seen = set()

    selectors = [
        "article h1 a[href]",
        "article h2 a[href]",
        "article h3 a[href]",
        ".recipe-card a[href]",
        ".recipe-item a[href]",
        ".recipe-post a[href]",
        ".recipes-list a[href]",
        ".entry-title a[href]",
        ".post-title a[href]",
        ".recipe a[href]",
        ".recipes a[href]",
    ]

    candidates = []

    for selector in selectors:
        candidates.extend(soup.select(selector))

    for anchor in candidates:
        href = anchor.get("href")

        if not href:
            continue

        recipe_url = normalize_url(href, page_url)

        if not recipe_url or recipe_url in seen:
            continue

        container = anchor.find_parent("article") or anchor.find_parent(
            ["li", "section", "div"]
        )

        if not is_recipe_candidate(recipe_url, from_recipe_card=True):
            continue

        if urlparse(recipe_url).path.rstrip("/") == urlparse(
            normalize_url(page_url)
        ).path.rstrip("/"):
            continue

        seen.add(recipe_url)

        title = clean_text(anchor.get_text(" ", strip=True))

        if not title and container:
            title_element = container.select_one(
                "h1, h2, h3, h4, .entry-title, .post-title"
            )

            if title_element:
                title = clean_text(
                    title_element.get_text(" ", strip=True)
                )

        image_element = container.select_one("img") if container else None
        image_url = ""

        if image_element:
            for attribute in (
                "src",
                "data-src",
                "data-lazy-src",
                "data-original",
                "data-lazy",
            ):
                value = image_element.get(attribute)

                if value and not value.startswith("data:"):
                    image_url = urljoin(page_url, value)
                    break

        recipes.append({
            "title": title,
            "url": recipe_url,
            "image_url": image_url,
            "image_alt": (
                clean_text(image_element.get("alt"))
                if image_element
                else ""
            ),
        })

    return recipes